# Installations

In [ ]:
!cp /kaggle/input/datasets-wheel/datasets-2.14.4-py3-none-any.whl /kaggle/working
!pip install  /kaggle/working/datasets-2.14.4-py3-none-any.whl
!cp /kaggle/input/backup-806/util_openbook.py .

In [ ]:
# installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

!pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

# Run Backup

In [ ]:
import ctypes
import gc
import pickle
import torch

libc = ctypes.CDLL("libc.so.6")

In [ ]:
from util_illimodel1 import get_contexts, generate_openbook_output
import pickle

get_contexts()
generate_openbook_output()

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

# Load libraries

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd 
from datasets import load_dataset, load_from_disk
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import LongformerTokenizer, LongformerForMultipleChoice
import transformers
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import unicodedata

from transformers import AutoTokenizer
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from torch.utils.data import DataLoader

import faiss
from sentence_transformers import SentenceTransformer

import os

# Read data

In [ ]:
backup_model_predictions = pd.read_csv("submission_backup.csv")

print(backup_model_predictions.shape)
backup_model_predictions.head(2)

In [ ]:
df_valid = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv")

print(df_valid.shape)
df_valid.head(2)

In [ ]:
stop_words = ['each', 'you', 'the', 'use', 'used',
                  'where', 'themselves', 'nor', "it's", 'how', "don't", 'just', 'your',
                  'about', 'himself', 'with', "weren't", 'hers', "wouldn't", 'more', 'its', 'were',
                  'his', 'their', 'then', 'been', 'myself', 're', 'not',
                  'ours', 'will', 'needn', 'which', 'here', 'hadn', 'it', 'our', 'there', 'than',
                  'most', "couldn't", 'both', 'some', 'for', 'up', 'couldn', "that'll",
                  "she's", 'over', 'this', 'now', 'until', 'these', 'few', 'haven',
                  'of', 'wouldn', 'into', 'too', 'to', 'very', 'shan', 'before', 'the', 'they',
                  'between', "doesn't", 'are', 'was', 'out', 'we', 'me',
                  'after', 'has', "isn't", 'have', 'such', 'should', 'yourselves', 'or', 'during', 'herself',
                  'doing', 'in', "shouldn't", "won't", 'when', 'do', 'through', 'she',
                  'having', 'him', "haven't", 'against', 'itself', 'that',
                  'did', 'theirs', 'can', 'those',
                  'own', 'so', 'and', 'who', "you've", 'yourself', 'her', 'he', 'only',
                  'what', 'ourselves', 'again', 'had', "you'd", 'is', 'other',
                  'why', 'while', 'from', 'them', 'if', 'above', 'does', 'whom',
                  'yours', 'but', 'being', "wasn't", 'be']

In [ ]:
!cp -r /kaggle/input/stem-wiki-cohere-no-emb /kaggle/working
!cp -r /kaggle/input/all-paraphs-parsed-expanded /kaggle/working/

# Process Docs functions

In [ ]:
def SplitList(mylist, chunk_size):
    return [mylist[offs:offs+chunk_size] for offs in range(0, len(mylist), chunk_size)]

def get_relevant_documents_parsed(df_valid):
    df_chunk_size=600
    paraphs_parsed_dataset = load_from_disk("/kaggle/working/all-paraphs-parsed-expanded")
    modified_texts = paraphs_parsed_dataset.map(lambda example:
                                             {'temp_text':
                                              f"{example['title']} {example['section']} {example['text']}".replace('\n'," ").replace("'","")},
                                             num_proc=2)["temp_text"]
    
    if len(df_valid) == 200: modified_texts = modified_texts[:10000]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         paraphs_parsed_dataset[idx.item()]["title"],
                         paraphs_parsed_dataset[idx.item()]["text"],
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



def get_relevant_documents(df_valid):
    df_chunk_size=800
    
    cohere_dataset_filtered = load_from_disk("/kaggle/working/stem-wiki-cohere-no-emb")
    modified_texts = cohere_dataset_filtered.map(lambda example:
                                             {'temp_text':
                                              unicodedata.normalize("NFKD", f"{example['title']} {example['text']}").replace('"',"")},
                                             num_proc=2)["temp_text"]
    
    if len(df_valid) == 200: modified_texts = modified_texts[:10000]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         cohere_dataset_filtered[idx.item()]["title"],
                         unicodedata.normalize("NFKD", cohere_dataset_filtered[idx.item()]["text"]),
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



# FAISS function

In [ ]:
# FAISS specs

SIM_MODEL = '/kaggle/input/sentencetransformers-allminilml6v2/sentence-transformers_all-MiniLM-L6-v2'
# SIM_MODEL = 'sentence-transformers/paraphrase-albert-small-v2'
DEVICE = 0
MAX_LENGTH = 384
BATCH_SIZE = 16

# Embedding model init
model = SentenceTransformer(SIM_MODEL, device='cuda')

In [ ]:
def build_faiss_index(texts):
        
    # Encode texts
    embeddings = model.encode(texts)
    dim = embeddings.shape[1]
    
    # Build FAISS index
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    
    return index

# Retrieval function

In [ ]:
def retrieval(df_valid, modified_texts):
    
    print("Embed Questions")
    
    corpus_df_valid = df_valid.apply(lambda row:
                                     f'{row["prompt"]}\n{row["prompt"]}\n{row["prompt"]}\n{row["A"]}\n{row["B"]}\n{row["C"]}\n{row["D"]}\n{row["E"]}',
                                     axis=1).values
    
    corpus_faiss = []

    for prompt in tqdm(corpus_df_valid):

        corpus_faiss.append(model.encode(prompt))

    print(f"Num Question Embeddings: {len(corpus_faiss)}")
    print(f"Question Embeddings shape: {corpus_faiss[0].shape}")

    chunk_size = 100000
    top_per_chunk = 10
    top_per_query = 10

    all_chunk_top_indices = []
    all_chunk_top_values = []
    
    index = build_faiss_index(modified_texts)


    for idx in tqdm(range(0, len(modified_texts), chunk_size)):

        chunk_top_values = []
        chunk_top_indices = []

        for query in tqdm(corpus_faiss):
            
            scores, inds = index.search(query.reshape(1, MAX_LENGTH), top_per_chunk)

            chunk_top_values.append(scores)
            chunk_top_indices.append(inds)

        chunk_top_values = np.vstack(chunk_top_values)
        chunk_top_indices = np.vstack(chunk_top_indices)

        all_chunk_top_indices.append(chunk_top_indices + idx)
        all_chunk_top_values.append(chunk_top_values)

        del chunk_top_values, chunk_top_indices

        _ = gc.collect()
        libc.malloc_trim(0)
        torch.cuda.empty_cache()
    
    del index

    _ = gc.collect()
    libc.malloc_trim(0)
    torch.cuda.empty_cache()

    top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
    top_values_array = np.concatenate(all_chunk_top_values, axis=1)

    merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
    merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
    articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]

    return articles_indices, merged_top_scores


# DataLoader

In [ ]:
def prepare_answering_input(
        tokenizer, 
        question,  
        options,   
        context,   
        max_seq_length=4096,
    ):
    c_plus_q   = context + ' ' + tokenizer.bos_token + ' ' + question
    c_plus_q_4 = [c_plus_q] * len(options)
    tokenized_examples = tokenizer(
        c_plus_q_4, options,
        max_length=max_seq_length,
        padding="longest",
        truncation=False,
        return_tensors="pt",
    )
    input_ids = tokenized_examples['input_ids'].unsqueeze(0)
    attention_mask = tokenized_examples['attention_mask'].unsqueeze(0)
    example_encoded = {
        "input_ids": input_ids.to(model.device.index),
        "attention_mask": attention_mask.to(model.device.index),
    }
    return example_encoded


# Process Docs

In [ ]:
%%capture

retrieved_articles_parsed = get_relevant_documents_parsed(df_valid)
gc.collect()

In [ ]:
%%capture

retrieved_articles = get_relevant_documents(df_valid)
gc.collect()

In [ ]:
retrieved_articles_parsed[0][0]

In [ ]:
retrieved_articles[0][-4][2]

# Model

In [ ]:
model_dir = "/kaggle/input/llmse-illift/debertav3-chkpt-1025-epoch-1-60k/checkpoint-1025/"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForMultipleChoice.from_pretrained(model_dir).cuda()

# Inference

In [ ]:
predictions = []
submit_ids = []

for index in tqdm(range(df_valid.shape[0])):
    columns = df_valid.iloc[index].values
    submit_ids.append(columns[0])
    question = columns[1]
    options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
    context1 = f"{retrieved_articles[index][-4][2]}\n{retrieved_articles[index][-3][2]}\n{retrieved_articles[index][-2][2]}\n{retrieved_articles[index][-1][2]}"
    context2 = f"{retrieved_articles_parsed[index][-3][2]}\n{retrieved_articles_parsed[index][-2][2]}\n{retrieved_articles_parsed[index][-1][2]}"
    inputs1 = prepare_answering_input(
        tokenizer=tokenizer, question=question,
        options=options, context=context1,
        )
    inputs2 = prepare_answering_input(
        tokenizer=tokenizer, question=question,
        options=options, context=context2,
        )
    
    with torch.no_grad():
        outputs1 = model(**inputs1)    
        losses1 = -outputs1.logits[0].detach().cpu().numpy()
        probability1 = torch.softmax(torch.tensor(-losses1), dim=-1)
        
    with torch.no_grad():
        outputs2 = model(**inputs2)
        losses2 = -outputs2.logits[0].detach().cpu().numpy()
        probability2 = torch.softmax(torch.tensor(-losses2), dim=-1)
        
    probability_ = (probability1 + probability2)/2

    if probability_.max() > 0.4:
        predict = np.array(list("ABCDE"))[np.argsort(probability_)][-3:].tolist()[::-1]
    else:
        predict = backup_model_predictions.iloc[index].prediction.replace(" ","")
    predictions.append(predict)

predictions = [" ".join(i) for i in predictions]

# Submission file

In [ ]:
pd.DataFrame({'id':submit_ids,'prediction':predictions}).to_csv('submission.csv', index=False)

# Testing

In [ ]:
# df_chunk_size=600
# paraphs_parsed_dataset = load_from_disk("/kaggle/working/all-paraphs-parsed-expanded")
# modified_texts = paraphs_parsed_dataset.map(lambda example:
#                                          {'temp_text':
#                                           f"{example['title']} {example['section']} {example['text']}".replace('\n'," ").replace("'","")},
#                                          num_proc=2)["temp_text"]

In [ ]:
# corpus_df_valid = df_valid.apply(lambda row:
#                                      f'{row["prompt"]}\n{row["prompt"]}\n{row["prompt"]}\n{row["A"]}\n{row["B"]}\n{row["C"]}\n{row["D"]}\n{row["E"]}',
#                                      axis=1).values

# len(corpus_df_valid)

In [ ]:
# %%capture

# corpus_faiss = []

# for prompt in tqdm(corpus_df_valid):

#     corpus_faiss.append(model.encode(prompt))

In [ ]:
# print(f"Num Question Embeddings: {len(corpus_faiss)}")
# print(f"Question Embeddings shape: {corpus_faiss[0].shape}")

In [ ]:
# chunk_size = 100000
# top_per_chunk = 10
# top_per_query = 10

# all_chunk_top_indices = []
# all_chunk_top_values = []

# print("Building FAISS index of Wiki Corpus")
# index = build_faiss_index(modified_texts[0: 0+chunk_size])

In [ ]:
# chunk_top_values = []
# chunk_top_indices = []

# for query in tqdm(corpus_faiss):
#     scores, inds = index.search(query.reshape(1, MAX_LENGTH), top_per_chunk)
    
#     chunk_top_values.append(scores)
#     chunk_top_indices.append(inds)

# chunk_top_values = np.vstack(chunk_top_values)
# chunk_top_indices = np.vstack(chunk_top_indices)

# print(chunk_top_values.shape)
# print(chunk_top_indices.shape)

In [ ]:
# corpus_faiss = []

# for prompt in tqdm(corpus_df_valid):

#     corpus_faiss.append(model.encode(prompt))
    
# print(f"Num Question Embeddings: {len(corpus_faiss)}")
# print(f"Question Embeddings shape: {corpus_faiss[0].shape}")

# chunk_size = 100000
# top_per_chunk = 10
# top_per_query = 10

# all_chunk_top_indices = []
# all_chunk_top_values = []


# for idx in tqdm(range(0, len(modified_texts), chunk_size)):
    
#     index = build_faiss_index(modified_texts[idx: idx+chunk_size])
    
#     chunk_top_values = []
#     chunk_top_indices = []

#     for query in tqdm(corpus_faiss):
#         scores, inds = index.search(query.reshape(1, MAX_LENGTH), top_per_chunk)

#         chunk_top_values.append(scores)
#         chunk_top_indices.append(inds)

#     chunk_top_values = np.vstack(chunk_top_values)
#     chunk_top_indices = np.vstack(chunk_top_indices)
    
#     all_chunk_top_indices.append(chunk_top_indices + idx)
#     all_chunk_top_values.append(chunk_top_values)
    
#     del index, chunk_top_values, chunk_top_indices

#     _ = gc.collect()
#     libc.malloc_trim(0)
#     torch.cuda.empty_cache()

# top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
# top_values_array = np.concatenate(all_chunk_top_values, axis=1)

# merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
# merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
# articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]

    

In [ ]:
# all_chunk_top_indices.append(chunk_top_indices + 0)
# all_chunk_top_values.append(chunk_top_values)

# # del index, chunk_top_values, chunk_top_indices

# _ = gc.collect()
# libc.malloc_trim(0)
# torch.cuda.empty_cache()

# top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
# top_values_array = np.concatenate(all_chunk_top_values, axis=1)

# merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
# merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
# articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]

In [ ]:
# print(articles_indices.shape)
# print(merged_top_scores.shape) 